In [1]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

### Step 1 - Indexing

#### Step 1.a - Document Ingestion

In [8]:
video_id = "Gfr50f6ZBvo"  # only the ID, not full URL

try:
    # Get English transcript
    api = YouTubeTranscriptApi()
    transcript_list = api.fetch(video_id)

    # Convert list of chunks to plain text
    transcript = "\n".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video")

the following is a conversation with
demus hasabis
ceo and co-founder of deepmind
a company that has published and builds
some of the most incredible artificial
intelligence systems in the history of
computing including alfred zero that
learned
all by itself to play the game of gold
better than any human in the world and
alpha fold two that solved protein
folding
both tasks considered nearly impossible
for a very long time
demus is widely considered to be one of
the most brilliant and impactful humans
in the history of artificial
intelligence and science and engineering
in general
this was truly an honor and a pleasure
for me to finally sit down with him for
this conversation and i'm sure we will
talk many times again in the future
this is the lex friedman podcast to
support it please check out our sponsors
in the description and now dear friends
here's demis
hassabis
let's start with a bit of a personal
question
am i an ai program you wrote to
interview people until i get good enough


#### Step 1.b - Text Splitting

In [9]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
chunks = splitter.create_documents([transcript])

In [10]:
len(chunks)

169

In [16]:
chunks[100].page_content

"can you explain this work and can ai\nmodel and simulate arbitrary quantum\nmechanical systems in the future yeah so\nthis is another problem i've had my eye\non for you know a decade or more which\nis um\nuh sort of simulating the properties of\nelectrons if you can do that you can\nbasically describe how elements and\nmaterials and substances work so it's\nkind of like fundamental if you want to\nadvance material science um and uh you\nknow we have schrodinger's equation and\nthen we have approximations to that\ndensity functional theory these things\nare you know are famous and um people\ntry and write approximations to to these\nuh uh to these functionals and and kind\nof come up with descriptions of the\nelectron clouds where they're gonna go\nhow they're gonna interact when you put\ntwo elements together uh and what we try\nto do is learn a simulation uh uh\nlearner functional that will describe\nmore chemistry types of chemistry so um\nuntil now you know you can run expensive"

#### Step 1c & 1d - Embedding Generation & Storing Embeddings Vector Store

In [17]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vector_store = FAISS.from_documents(chunks, embedding_model)

In [18]:
vector_store.index_to_docstore_id

{0: 'aed558ae-c402-4329-8d8f-5a637f6949ac',
 1: '7b59dea0-4098-416d-afb3-9ec39b338fb3',
 2: 'bff21185-fd39-4043-847d-ecd260b4ba91',
 3: '4e9bdfa2-2ac9-4fa5-83a8-04903a5cfea0',
 4: '140c9bed-7e7c-4229-874c-0c6b8a1b8763',
 5: 'ede88dfd-6073-4d49-ac03-28998e44ca33',
 6: '1a3d3c9a-6afa-497c-b657-4815f8777488',
 7: 'c7875807-2bbc-44e7-b9f3-3ed44b713a8d',
 8: '2c54f196-cb82-49f4-940d-41c16e5fe759',
 9: 'a60e9051-1716-49b2-8543-997efc432ea6',
 10: 'd72853ff-f00e-4003-98dd-628e15075c58',
 11: 'a82bd333-e03e-4440-baa5-5d4fd12ef8e6',
 12: 'd23bf3c0-8d56-432a-b819-51b58dd9d823',
 13: '8b20d690-a11d-447a-a916-bc9eaf89ab55',
 14: '55ba2cf1-b3cb-4d21-9f10-bd18d5ae394c',
 15: '5269a9fe-8377-4001-8cd3-a059e4e459fa',
 16: '160415cc-150a-4b45-b57d-636bc22cbc1e',
 17: 'f2047300-be1c-442a-b965-bfb72eac4577',
 18: 'cb2929d2-65ae-4860-921a-5e49c695c33b',
 19: '77c100f6-97e0-4f1a-9300-39b3357a1d1e',
 20: '8333942f-b01b-4318-a20a-41a07a1370bf',
 21: '0c01407a-1aba-4072-a587-4f69b6754ba4',
 22: 'e067f3e7-d09c-

In [22]:
vector_store.get_by_ids(['0779d915-a61c-44a5-88cc-286bcbc04d9c'])

[Document(id='0779d915-a61c-44a5-88cc-286bcbc04d9c', metadata={}, page_content="is not a game that could be solved\nbecause of the combinatorial complexity\nit's just too it's it's it's you know\nno matter how much moore's law you have\ncompute is just never going to be able\nto crack the game of go yeah and so that\nthen there's something compelling about\nfacing sort of taking on the\nimpossibility of that task from the\nai\nresearcher perspective engineer\nperspective and then as a human being\njust observing this whole thing\nyour\nbeliefs about what you thought was\nimpossible\nbeing broken apart\nit's it's uh humbling\nto realize we're not as smart as we\nthought\nit's humbling to realize that the things\nwe think are impossible now perhaps will\nbe done\nin the future there's something\nreally powerful about a game ai system\nbeing a human being in a game that\ndrives that message\nuh home for like millions billions of\npeople especially in the case of go sure\nwell look i think

### Step 2 - Retriever

In [23]:
retriever = vector_store.as_retriever(search_type = 'similarity', search_kwargs = {'k':4})

In [24]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x109d92190>, search_kwargs={'k': 4})

In [25]:
retriever.invoke('What is deepmind')

[Document(id='761a9d3f-9df1-41d6-b1a2-b8860c11a523', metadata={}, page_content="going in the right direction so so that\nthat was one reason we pushed so hard on\nthat and that's and just going back to\nyour early question about organization\nthe other big thing that i think we\ninnovated with at deepmind to encourage\ninvention and and uh and innovation was\nthe multi-disciplinary organization we\nbuilt and we still have today so\ndeepmind originally was a confluence of\nthe of the most cutting-edge knowledge\nin neuroscience with machine learning\nengineering and mathematics right and\nand gaming\nand then since then we built that out\neven further so we have philosophers\nhere and and uh by you know ethicists\nbut also other types of scientists\nphysicists and so on um and that's what\nbrings together i tried to build a sort\nof um new type of bell labs but in this\ngolden era right uh\nand and a new expression of that um to\ntry and uh foster this incredible sort\nof innovation mac

### Step 3 - Augmentation

In [26]:
model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

In [27]:
prompt = PromptTemplate(
    template="""
        You are a helpful assistant.
        Answer ONLY from the provided transcript context.
        If the context is insufficient, just say you don't know.

        {context}
        Question:{question}
    """,
    input_variables=['context', 'question']
)

In [33]:
question = '"is the topic of nuclear fusion discussed in this video? if yes then what was discussed'
retrieved_docs = retriever.invoke(question)

In [34]:
content_text = '\n\n'.join(doc.page_content for doc in retrieved_docs)

In [35]:
final_prompt = prompt.invoke({'context':content_text, 'question': question})

### Step 4 - Generation

In [36]:
answer = model.invoke(final_prompt)
print(answer.content)

Yes, the topic of nuclear fusion is discussed in this video. 

The discussion revolves around the application of AI in solving the challenges of nuclear fusion, specifically in the area of plasma control. The speaker mentions that they collaborated with EPFL in Switzerland to use their test reactor and developed a deep reinforcement learning (DRL) model to control high-temperature plasmas in a tokamak. They also mention that they published a paper on this work in Nature and were able to hold the plasma in specific shapes for a record amount of time.


### Building a Chain

In [37]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [38]:
def format_docs(retrieved_docs):
    content_text = '\n\n'.join(doc.page_content for doc in retrieved_docs)
    return content_text

In [39]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [42]:
parallel_chain.invoke('who is Demus')

{'context': "the following is a conversation with\ndemus hasabis\nceo and co-founder of deepmind\na company that has published and builds\nsome of the most incredible artificial\nintelligence systems in the history of\ncomputing including alfred zero that\nlearned\nall by itself to play the game of gold\nbetter than any human in the world and\nalpha fold two that solved protein\nfolding\nboth tasks considered nearly impossible\nfor a very long time\ndemus is widely considered to be one of\nthe most brilliant and impactful humans\nin the history of artificial\nintelligence and science and engineering\nin general\nthis was truly an honor and a pleasure\nfor me to finally sit down with him for\nthis conversation and i'm sure we will\ntalk many times again in the future\nthis is the lex friedman podcast to\nsupport it please check out our sponsors\nin the description and now dear friends\nhere's demis\nhassabis\nlet's start with a bit of a personal\nquestion\nam i an ai program you wrote t

In [43]:
parser = StrOutputParser()

In [44]:
main_chain = parallel_chain | prompt | model | parser

In [45]:
main_chain.invoke('Can you summarise the video')

"The conversation appears to be a discussion between two individuals, possibly a game designer and a physicist, about various topics including the nature of explanation, biology, and game design.\n\nThe conversation starts with a discussion about explaining complex concepts, where the physicist suggests that a deeper understanding of physics could provide glimpses into the nature of consciousness, life, and gravity.\n\nThe conversation then shifts to a discussion about biology, where the physicist compares it to a cellular automata, suggesting that it's too complicated to figure out the rules and requires learning through simulation.\n\nThe game designer then mentions open-sourcing their work, including the AlphaFold project and the Mijoko physics simulation engine.\n\nThe conversation then turns to a discussion about chess, where the game designer suggests that the dynamic nature of chess positions, particularly the balance between the bishop and knight, is a critical reason for the g